# 文件读取
文件读取的核心任务是从化学文件中提取出计算分子性质时需要的信息，主要分为三类

- 结构信息
- 基组信息
- 轨道信息

所有的读取器都包含在`pywfn.reader`中，当前支持的读取器及支持的信息如下：

|读取类|结构信息|基组信息|轨道信息|说明|
|:---:|:---:|:---:|:---:|:---:|
|`LogReader`|√|?|?|读取高斯的输出文件|
|`GjfReader`|√|×|×|读取高斯的输入文件|
|`FchReader`|√|√|√|读取高斯的检查点文件|
|`ModReader`|√|√|√|读取`.molden`文件|
|`XyzReader`|√|×|×|读取`.sdf`文件|
|`MolReader`|√|×|×|读取`.mol`文件|
|`SdfReader`|√|×|×|读取`.sdf`文件|
|`AnyReader`|?|?|?|可自定义信息|

## 信息详情

### 结构信息

包含原子的`类型`和`坐标`，以常见的高斯输入文件类型`gjf`为例

```
%chk=D:\gfile\C6H6.chk
# b3lyp/6-31g(d) pop=full gfinput iop(3/33=1)

Title Card Required

0 1
 C                  0.00000000    1.40140000   -0.00000000
 C                 -1.21364800    0.70070000   -0.00000000
 C                 -1.21364800   -0.70070000   -0.00000000
 C                  0.00000000   -1.40140000   -0.00000000
 C                  1.21364800   -0.70070000   -0.00000000
 C                  1.21364800    0.70070000   -0.00000000
 H                  0.00000000    2.47140000   -0.00000000
 H                 -2.14029518    1.23570000   -0.00000000
 H                 -2.14029518   -1.23570000   -0.00000000
 H                  0.00000000   -2.47140000   -0.00000000
 H                  2.14029518   -1.23570000   -0.00000000
 H                  2.14029518    1.23570000   -0.00000000


```

其中存储着原子的元素符号，即为原子类型，以及原子的xyz坐标，即原子坐标。

不同类型的文件存储使用的原子坐标的单位不同，可以为`Angstorm`或`Borh`，pywfn内部统一转为`Borh`单位。

### 基组信息
包含高斯型基函数的系数和指数

以`6-31g(d)`基组为例：

<!-- ![基组示例](./imgs/基组示例.png) -->

```
      1 0
 S   6 1.00       0.000000000000
      0.3047524880D+04  0.1834737132D-02
      0.4573695180D+03  0.1403732281D-01
      0.1039486850D+03  0.6884262226D-01
      0.2921015530D+02  0.2321844432D+00
      0.9286662960D+01  0.4679413484D+00
      0.3163926960D+01  0.3623119853D+00
 SP   3 1.00       0.000000000000
      0.7868272350D+01 -0.1193324198D+00  0.6899906659D-01
      0.1881288540D+01 -0.1608541517D+00  0.3164239610D+00
      0.5442492580D+00  0.1143456438D+01  0.7443082909D+00
 SP   1 1.00       0.000000000000
      0.1687144782D+00  0.1000000000D+01  0.1000000000D+01
 D   1 1.00       0.000000000000
      0.8000000000D+00  0.1000000000D+01
```


- `1`行中的`1`代表当前基组信息属于第一个原子
- `2`行中`S`代表`1S`轨道，`6`代表`1S`轨道有`6`个GTO函数线性组合而成
- `3-8`中两列数据分别代表GTO函数的`指数`和`系数`
- `9`行中的`SP`代表`2S`和`2P`轨道，分别也是有`6`个GTO函数线性组合而成
- `10-12`中，第一列和第二列为`2S`和`2P`的GTO指数，第三列代表两者共享的系数
- 其它的也都是类似的

### 轨道信息


轨道信息主要就是与分子轨道相关的信息，主要包含：

- **轨道系数矩阵**。分为开窍层和闭壳层。形状为[nato,nobt]，对于闭壳层，nato=nobt；对于开窍层，nato*2=nobt
- **分子轨道-能量**。长度为nobt的浮点数列表。
- **分子轨道-占据**。长度为nobt的布尔值列表。表示分子轨道是否占据
- **原子轨道-原子索引**。每个原子轨道对应的原子索引
- **原子轨道-轨道壳层**。原子轨道对应的壳层索引。
- **原子轨道-轨道符号**。每个原子轨道的轨道符号。可以为：`S`、`PX`、`PY`、`PZ` 等等

这些信息都存储在`base.Coef`类型中

直接读取到的原子轨道信息混合了`笛卡尔型基函数`和`球谐型基函数`，在Coef类型中可以将转为全部为笛卡尔型或者全部为球谐型。

## fch文件

**示例代码**

In [7]:
from pywfn.base import Mole
mole=Mole.from_file('./mols/C6H6.fch')
print(mole)

  0  1
atm             x             y             z       crg
  C   -0.00000000    2.64826219    0.00000000    6.0000
  C    2.29346233    1.32413110    0.00000000    6.0000
  C    2.29346233   -1.32413109    0.00000000    6.0000
  C    0.00000000   -2.64826219    0.00000000    6.0000
  C   -2.29346233   -1.32413110    0.00000000    6.0000
  C   -2.29346233    1.32413109    0.00000000    6.0000
  H   -0.00000000    4.67026914    0.00000000    1.0000
  H    4.04457172    2.33513457    0.00000000    1.0000
  H    4.04457172   -2.33513457    0.00000000    1.0000
  H    0.00000000   -4.67026914    0.00000000    1.0000
  H   -4.04457172   -2.33513457    0.00000000    1.0000
  H   -4.04457172    2.33513457    0.00000000    1.0000
idx  atm  shl  ang        form
0    0    0    0           sph(6)
       3.047525E3   1.834737E-3
       4.573695E2   1.403732E-2
       1.039487E2   6.884262E-2
       2.921016E1   2.321844E-1
       9.286663E0   4.679413E-1
       3.163927E0   3.623120E-1
1    0  

In [8]:
from pywfn.reader import FchReader
reader=FchReader('./mols/C6H6.fch')

## gjf文件

**示例代码**

In [9]:
from pywfn.base import Mole
mole=Mole.from_file("./mols/C6H6.gjf")
print(mole)

  0  1
atm             x             y             z       crg
  C    0.00000000    2.64826219    0.00000000    6.0000
  C    2.29346233    1.32413110    0.00000000    6.0000
  C    2.29346233   -1.32413110    0.00000000    6.0000
  C    0.00000000   -2.64826219    0.00000000    6.0000
  C   -2.29346233   -1.32413110    0.00000000    6.0000
  C   -2.29346233    1.32413110    0.00000000    6.0000
  H    0.00000000    4.67026915    0.00000000    1.0000
  H    4.04457138    2.33513457    0.00000000    1.0000
  H    4.04457138   -2.33513457    0.00000000    1.0000
  H    0.00000000   -4.67026915    0.00000000    1.0000
  H   -4.04457138   -2.33513457    0.00000000    1.0000
  H   -4.04457138    2.33513457    0.00000000    1.0000



gjf文件仅包含结构信息，所以打印分子的时候只包含结构信息

## log文件
高斯的输出文件，后缀可以为`.log`或`.out`，进行波函数分析时需要加入关键词：`pop=full gfinput iop(3/33=1)`

其中

- `pop=full` 用于打印完整的分子轨道系数矩阵
- `gfinput` 用于打印 基组信息
- `iop(3/33=1)` 用于打印 重叠矩阵 等

**示例代码**

In [10]:
from pywfn.base import Mole
mole=Mole.from_file('./mols/C6H6.out')
print(mole)

  0  1
atm             x             y             z       crg
  C    0.00000000    2.64826219    0.00000000    6.0000
  C    2.29346233    1.32413110    0.00000000    6.0000
  C    2.29346233   -1.32413110    0.00000000    6.0000
  C    0.00000000   -2.64826219    0.00000000    6.0000
  C   -2.29346233   -1.32413110    0.00000000    6.0000
  C   -2.29346233    1.32413110    0.00000000    6.0000
  H    0.00000000    4.67026915    0.00000000    1.0000
  H    4.04457138    2.33513457    0.00000000    1.0000
  H    4.04457138   -2.33513457    0.00000000    1.0000
  H    0.00000000   -4.67026915    0.00000000    1.0000
  H   -4.04457138   -2.33513457    0.00000000    1.0000
  H   -4.04457138    2.33513457    0.00000000    1.0000
idx  atm  shl  ang        form
0    0    0    0           sph(6)
       3.047525E3   1.834737E-3
       4.573695E2   1.403732E-2
       1.039487E2   6.884262E-2
       2.921016E1   2.321844E-1
       9.286663E0   4.679413E-1
       3.163927E0   3.623120E-1
1    0  

## molden文件
一种存储波函数信息的常见文件类型

**示例代码**

In [11]:
from pywfn.base import Mole
mole=Mole.from_file('./mols/C6H6.molden')
print(mole)

  0  1
atm             x             y             z       crg
  C    2.63419900    0.15831100   -0.00000200    6.0000
  C    1.17993700    2.36041300   -0.00010500    6.0000
  C   -1.45410500    2.20201800    0.00008400    6.0000
  C   -2.63419600   -0.15841400   -0.00001700    6.0000
  C   -1.18002700   -2.36037000   -0.00008200    6.0000
  C    1.45419200   -2.20196000    0.00007800    6.0000
  H    4.68353600    0.28155300    0.00004900    1.0000
  H    2.09799900    4.19674900   -0.00005100    1.0000
  H   -2.58557000    3.91513200    0.00017000    1.0000
  H   -4.68355000   -0.28139000    0.00003300    1.0000
  H   -2.09785900   -4.19682300   -0.00005900    1.0000
  H    2.58544000   -3.91521400    0.00012300    1.0000
idx  atm  shl  ang        form
0    0    0    0           sph(6)
       3.047525E3   6.276294E-6
       4.573695E2   1.991470E-4
       1.039487E2   2.967106E-3
       2.921016E1   2.592821E-2
       9.286663E0   1.234203E-1
       3.163927E0   2.142904E-1
1    0  

## mol文件
存储分子结合结构的文件

**示例代码**

In [12]:
from pywfn.base import Mole
mole=Mole.from_file('./mols/C6H6.mol')
print(mole)

  0  1
atm             x             y             z       crg
  C    0.00000000    1.40140000    0.00000000    6.0000
  C    1.21360000    0.70070000    0.00000000    6.0000
  C    1.21360000   -0.70070000    0.00000000    6.0000
  C    0.00000000   -1.40140000    0.00000000    6.0000
  C   -1.21360000   -0.70070000    0.00000000    6.0000
  C   -1.21360000    0.70070000    0.00000000    6.0000
  H    0.00000000    2.47140000    0.00000000    1.0000
  H    2.14030000    1.23570000    0.00000000    1.0000
  H    2.14030000   -1.23570000    0.00000000    1.0000
  H    0.00000000   -2.47140000    0.00000000    1.0000
  H   -2.14030000   -1.23570000    0.00000000    1.0000
  H   -2.14030000    1.23570000    0.00000000    1.0000



## sdf文件
包含分子几何结构信息的文件类型

**示例代码**

In [13]:
from pywfn.base import Mole
mole=Mole.from_file('./mols/C6H6.sdf')
print(mole)

  0  1
atm             x             y             z       crg
  C    0.00000000    1.40140000    0.00000000    6.0000
  C    1.21360000    0.70070000    0.00000000    6.0000
  C    1.21360000   -0.70070000    0.00000000    6.0000
  C    0.00000000   -1.40140000    0.00000000    6.0000
  C   -1.21360000   -0.70070000    0.00000000    6.0000
  C   -1.21360000    0.70070000    0.00000000    6.0000
  H    0.00000000    2.47140000    0.00000000    1.0000
  H    2.14030000    1.23570000    0.00000000    1.0000
  H    2.14030000   -1.23570000    0.00000000    1.0000
  H    0.00000000   -2.47140000    0.00000000    1.0000
  H   -2.14030000   -1.23570000    0.00000000    1.0000
  H   -2.14030000    1.23570000    0.00000000    1.0000



## xyz文件
包含分子结构信息的文件类型

`.xyz`文件可以包含多个结构，pywfn默认读取最后一个结构

**示例代码**

In [14]:
from pywfn.base import Mole
mol=Mole.from_file('./mols/C6H6.xyz')
print(mole)

  0  1
atm             x             y             z       crg
  C    0.00000000    1.40140000    0.00000000    6.0000
  C    1.21360000    0.70070000    0.00000000    6.0000
  C    1.21360000   -0.70070000    0.00000000    6.0000
  C    0.00000000   -1.40140000    0.00000000    6.0000
  C   -1.21360000   -0.70070000    0.00000000    6.0000
  C   -1.21360000    0.70070000    0.00000000    6.0000
  H    0.00000000    2.47140000    0.00000000    1.0000
  H    2.14030000    1.23570000    0.00000000    1.0000
  H    2.14030000   -1.23570000    0.00000000    1.0000
  H    0.00000000   -2.47140000    0.00000000    1.0000
  H   -2.14030000   -1.23570000    0.00000000    1.0000
  H   -2.14030000    1.23570000    0.00000000    1.0000

